# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIRʲ dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant (if you have not installed it already)
!pip install mlcroissant

## 1. Data Loading
We'll load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# The metadata attribute exposes schema information
metadata = dataset.metadata

# Print dataset title and description
print(f"{getattr(metadata, 'name')}: {getattr(metadata, 'description')}")

## 2. Data Overview
We review available record sets, fields, and their `@id` references.

In [ ]:
# Inspect record sets. 
# First, get all available record sets with their `@id` and field information.
record_sets = dataset.record_sets
print('All Record Sets:')
for rs in record_sets:
    print(f"- @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    print("  Fields:")
    for field in rs.get('field', []):
        print(f"    - @id: {field['@id']} | Name: {field.get('name', 'N/A')}")
    print()
# Preview records in each record set using their @id
for rs in record_sets:
    print(f"Sample records for record set {rs['@id']}:")
    try:
        for i, record in enumerate(dataset.records(record_set=rs['@id'])):
            print(record)
            if i >= 2:
                break
    except Exception as e:
        print("  Could not load records: ", str(e))
    print()

## 3. Data Extraction
We load data from one or more record sets into Pandas DataFrames for subsequent analysis.

All record sets and fields are referenced by their `@id`.

In [ ]:
# List of available record_set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print("Record Set @ids:", record_set_ids)

# Load all record set data
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for record set @id {rs_id}, shape: {df.shape}")

# Preview the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print('Columns:', dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let us process the extracted data: filter records, normalize numeric fields, and group by key attributes.

If record set fields are not numeric, please adjust selections accordingly.

In [ ]:
# Identify numeric fields from the first record set

first_rs_id = record_set_ids[0]
df = dataframes[first_rs_id]
numeric_columns = df.select_dtypes(include=['int', 'float']).columns
print('Numeric field @ids:', numeric_columns.tolist())

# Pick first numeric field for analysis
numeric_field_id = numeric_columns[0] if len(numeric_columns) > 0 else None
if numeric_field_id is not None:
    # Example: Filtering where the numeric value > 10
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization (z-score)
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized values for field {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouping by a categorical column if exists
    group_candidates = df.select_dtypes(include=['object', 'category']).columns
    group_field_id = group_candidates[0] if len(group_candidates) > 0 else None
    if group_field_id is not None:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped.head())
    else:
        print("No categorical fields available for grouping.")
else:
    print("No numeric fields available in the record set.")

## 5. Visualization
Let's visualize the distribution of the numeric field and grouped means.

We use matplotlib for basic visualization. Adjust field IDs and groups as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15, color='steelblue')
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id} in Record Set {first_rs_id}')
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(8,4))
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        grouped.plot(kind='bar', color='coral')
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xlabel(group_field_id)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIRʲ clinicopathological colorectal cancer dataset via its Croissant schema, performed basic metadata and schema overview, then extracted records and performed simple EDA and visualization.

- All entities were referenced by their `@id`, ensuring reproducibility and proper schema tracking.
- Data extraction and transformation are flexible using mlcroissant.
- Additional analyses and visualizations can be performed using the extracted DataFrames.

For more advanced Croissant and FAIRʲ dataset usage, explore:
- [mlcroissant documentation](https://github.com/mlcommons/croissant)
- [SenScience FAIRʲ registry](https://sen.science/doi/10.71728/senscience.qs2f-h81p)

Feel free to adapt this notebook for your research and exploration!